# 06 - Task 7: pairwise judging with Kaggle models
Create this notebook as a **utility benchmark** notebook so `kaggle_benchmarks` model-proxy calls are available. Attach outputs of notebooks 04 and 05. No GPU needed. Judging only reads saved pair files.

In [ ]:
# Private repo: add a GitHub token as a Kaggle secret named GITHUB_TOKEN
# (Add-ons -> Secrets -> Add secret, then tick it for this notebook).
REPO_URL = "https://github.com/zainabusman22/advanced-genai-pa2.git"
REPO = "/kaggle/working/advanced-genai-pa2"

import os, subprocess, sys
if not os.path.exists(REPO):
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    authed = REPO_URL.replace("https://", f"https://x-access-token:{token}@")
    subprocess.run(["git", "clone", "--depth", "1", authed, REPO], check=True, capture_output=True)
    subprocess.run(["git", "-C", REPO, "remote", "set-url", "origin", REPO_URL], check=True)  # don't keep the token on disk
    del token, authed
os.chdir(REPO)
import glob, shutil
import kaggle_benchmarks as kbench
print(list(kbench.llms.keys()) if hasattr(kbench.llms, "keys") else kbench.llms)

In [ ]:
# Copy saved pair files (and any earlier judgments, so reruns resume) into the repo.
os.makedirs("outputs/eval", exist_ok=True)
for f in glob.glob("/kaggle/input/**/outputs/eval/*.jsonl", recursive=True):
    dst = os.path.join("outputs/eval", os.path.basename(f))
    if not os.path.exists(dst):
        shutil.copy(f, dst)
!ls -l outputs/eval

In [ ]:
JUDGE = "google/gemini-3.6-flash"   # record the model actually used in the report
for name in ["uf_dpo_vs_sft", "hh_dpo_vs_sft", "persona_lora_vs_instruct", "sft_vs_base_alpaca"]:
    if not os.path.exists(f"outputs/eval/{name}_pairs.jsonl"):
        print("missing", name); continue
    !python scripts/judge_pairwise_kaggle.py --pairs_path outputs/eval/{name}_pairs.jsonl \
        --output_path outputs/eval/{name}_judged.jsonl --judge_model {JUDGE} \
        --batch_size 5 --sleep_seconds 5 --max_retries 3 --reuse_judgments

In [ ]:
!python scripts/summarize_results.py --eval_dir outputs/eval --out report/results_table.md
!cat report/results_table.md

In [ ]:
# Archive outputs so they survive the session (then: Save Version -> attach as dataset).
!cd {REPO} && zip -qr /kaggle/working/judge_outputs.zip outputs/eval report/results_table.md
!ls -lh /kaggle/working/*.zip